In [12]:
from os import environ
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage, SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import AzureChatOpenAI
from typing_extensions import Annotated, TypedDict
import pypandoc

In [13]:
import fitz  # PyMuPDF
from PIL import Image
import io
import json

In [14]:
import os
from langchain_groq import ChatGroq

groq_api_key = os.getenv('GROQ_API_KEY')
groq_model = "openai/gpt-oss-20b"

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name=groq_model,
)


In [15]:
# Define the custom class for a PDF file guide for general use
class PDFFile:
    def __init__(self, title):
        self.title = title
        self.pages = []

    def add_step(self, step_number, description, image_path):
        self.pages.append({
            "step_number": step_number,
            "description": description,
            "image_path": image_path
        })

    def export_pdf(self, output_path):
        doc = fitz.open()
        for page in self.pages:
            pdf_page = doc.new_page()
            text = f"Step {page['step_number']}: {page['description']}"
            pdf_page.insert_text((50, 50), text, fontsize=12)
            try:
                img_rect = fitz.Rect(50, 100, 400, 400)
                pdf_page.insert_image(img_rect, filename=page['image_path'])
            except Exception as e:
                pdf_page.insert_text((50, 420), f"Image not found: {page['image_path']}", fontsize=10)
        doc.save(output_path)



In [16]:

# JSON string defining the steps
json_string = '''
[
    {
        "step_number": 1,
        "description": "Unpack all components and lay them out.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 2,
        "description": "Organize screws, bolts, and tools.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 3,
        "description": "Attach legs to the tabletop.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 4,
        "description": "Secure legs with crossbars (if provided).",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 5,
        "description": "Tighten all screws and bolts.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 6,
        "description": "Flip the table upright and check for balance.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 7,
        "description": "Clean the surface and enjoy your new table.",
        "image_path": "b24486d014.png"
    }
]
'''


In [17]:
request = """we're participating in a hackaton event about gen ai. 
Our project is about creating an app that implements 
- generative ai
- langgraph
- react
- tailwind
- rag architecture

to generate different kind of files that can be used by new joiners or new team members for self study.

please generate a series of steps to follow to create the project structure and the main components needed for the app."""

output = llm.invoke(request)
print(output.content)

Below is a **step‑by‑step blueprint** that takes you from a blank repo to a working prototype that:

* Uses **LangGraph** to orchestrate a Retrieval‑Augmented Generation (RAG) workflow  
* Calls a generative LLM (OpenAI/Anthropic/…) to produce the content  
* Exposes a **React + Tailwind** UI that lets a new joiner pick a topic, generate a file, preview it and download it  

Feel free to cherry‑pick the parts you want to keep, rename folders, or swap out libraries – the outline is intentionally modular.

---

## 1️⃣ Project Skeleton

```
my-onboarding-gen/
├─ backend/          # Node + LangGraph / LangChain
│  ├─ src/
│  │  ├─ graph/
│  │  │  ├─ ingest.ts
│  │  │  ├─ query.ts
│  │  │  ├─ generate.ts
│  │  │  └─ index.ts   # LangGraph definition
│  │  ├─ routes/
│  │  │  └─ api.ts     # Express/FastAPI endpoints
│  │  ├─ utils/
│  │  │  └─ vectorStore.ts
│  │  └─ app.ts        # Express/FastAPI bootstrap
│  ├─ .env
│  ├─ package.json
│  └─ tsconfig.json
├─ frontend/          # React + V

In [18]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    pdf_json: str | None
    markdown: str | None
    pdf_file: str | None
    intent: str | None

In [19]:
def generate_pdf(state: State):
    """Converts markdown content to a pdf file using Pandoc."""
    output_file = "pdffile.pdf"

    pypandoc.convert_text(
        state.get("markdown", ""),
        "pdf",          # formato de salida
        format="md",     # formato de entrada
        outputfile=output_file,
        extra_args=["--standalone"]  # asegura un archivo completo
    )
    state["pdf_file"] = output_file
    return {"pdf_file": output_file}
#add pdf language
#check columns and layout
#translate prompts to english
#check templates
#check images
#llm_with_tools = llm.bind_tools([generate_pdf])

In [20]:
request = """we're participating in a hackaton event about gen ai. 
Our project is about creating an app that implements 
- generative ai
- langgraph
- react
- tailwind
- rag architecture

to generate different kind of files that can be used by new joiners or new team members for self study.

please generate a list of 15 smart and cool names that we can use in the hackaton for naming our project"""

output = llm.invoke(request)
print(output.content)

1. **OnboardAI**  
2. **GraphGuide**  
3. **RAG‑Ready Docs**  
4. **React‑RAG Navigator**  
5. **Tailwind Tutor**  
6. **GenLearn Hub**  
7. **SkillGraph Studio**  
8. **AutoMentor**  
9. **LearnLoop AI**  
10. **Onboarding Oracle**  
11. **Knowledge Kube**  
12. **DocuGraph Dynamo**  
13. **Genie‑Docs**  
14. **Tailored Trailblazer**  
15. **SparkSheet AI**


In [21]:

from langchain_core.prompts import PromptTemplate

def generate_pdf_request(context: str):
    "generate a sample request for creating a pdf file using reportlab library"

    prompt = """You are a smart assistant than can create payload samples for creating pdf files.
        The content of the sample must be extracted from this context:

        ### Context ###
        {context}
        Your job is to generate a sample request following the example below.
        

        The title field is the title of the document.
        The Text is the content of the document.
        The Document must be structured in pages.
        
        
        Please generate a sample request based on the context above. 

        ### Example of a request ###
    {{
    [
    {
        "step_number": 1,
        "description": "Unpack all components and lay them out.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 2,
        "description": "Organize screws, bolts, and tools.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 3,
        "description": "Attach legs to the tabletop.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 4,
        "description": "Secure legs with crossbars (if provided).",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 5,
        "description": "Tighten all screws and bolts.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 6,
        "description": "Flip the table upright and check for balance.",
        "image_path": "b24486d014.png"
    },
    {
        "step_number": 7,
        "description": "Clean the surface and enjoy your new table.",
        "image_path": "b24486d014.png"
    }
    ]
    }}   
    ### End of Example ###
        Return just the sample payload. Nothing else.
        """

    prompt_template = PromptTemplate.from_template(prompt)
    chain  = prompt_template | llm
    sample_request = chain.invoke({"context": context})
    return sample_request

In [22]:
### Generate Markdown


def generate_markdown(state: State):
    prompt = f"""
You are a content transformer.
You will receive a JSON that describes a slide presentation.
You must convert it to valid Markdown, using syntax that Pandoc can process to generate a PDF file.

### Instructions:
1. Don't invent content. Use only the information given in the JSON.
2. Don't explain anything, your output must be only Markdown.

### Example of Markdown syntax for PDF:

---
title: Pandoc User's Guide
author: John MacFarlane
date: 2025-10-05
---

# Synopsis

`pandoc` [*options*] [*input-file*]...

# Description

Pandoc is a [Haskell] library for converting from one markup format to
another, and a command-line tool that uses this library.

Pandoc can convert between numerous markup and word processing formats,
including, but not limited to, various flavors of [Markdown], [HTML],
[LaTeX] and [Word docx]. For the full lists of input and output formats,
see the `--from` and `--to` [options below][General options].
Pandoc can also produce [PDF] output: see [creating a PDF], below.

Pandoc's enhanced version of Markdown includes syntax for [tables],
[definition lists], [metadata blocks], [footnotes], [citations], [math],
and much more.  See below under [Pandoc's Markdown].

Pandoc has a modular design: it consists of a set of readers, which parse
text in a given format and produce a native representation of the document
(an _abstract syntax tree_ or AST), and a set of writers, which convert
this native representation into a target format. Thus, adding an input
or output format requires only adding a reader or writer. Users can also
run custom [pandoc filters] to modify the intermediate AST.

Because pandoc's intermediate representation of a document is less
expressive than many of the formats it converts between, one should
not expect perfect conversions between every format and every other.
Pandoc attempts to preserve the structural elements of a document, but
not formatting details such as margin size.  And some document elements,
such as complex tables, may not fit into pandoc's simple document
model.  While conversions from pandoc's Markdown to all formats aspire
to be perfect, conversions from formats more expressive than pandoc's
Markdown can be expected to be lossy.

"""

In [23]:
import json
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def create_pdf_from_json_fixed(json_str, output_path):
    """
    Creates a PDF file using reportlab from a JSON string with 'title' and 'text' fields.
    Example JSON:
    {
        "title": "My PDF Title",
        "text": "This is the main content of the PDF."
    }
    """
    data = json.loads(json_str)
    title = data.get("title", "Untitled")
    text = data.get("text", "")

    c = canvas.Canvas(output_path, pagesize=letter)
    width, height = letter

    c.setFont("Helvetica-Bold", 20)
    c.drawString(72, height - 72, title)

    c.setFont("Helvetica", 12)
    c.drawString(72, height - 100, text)

    c.save()

# Example usage:
# json_input = '{"title": "Demo PDF", "text": "This is a sample PDF generated from JSON."}'
# create_pdf_from_json(json_input, "output.pdf")

In [24]:
sample = generate_pdf_request("there are several animals in the jungle, such as the lion, tiger and elephant. funny huh ?")

KeyError: 'Input to PromptTemplate is missing variables {\'\\n        "step_number"\'}.  Expected: [\'\\n        "step_number"\', \'context\'] Received: [\'context\']\nNote: if you intended {\n        "step_number"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{\n        "step_number"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [ ]:
def clean_request(request:str):
        return request.replace("python","").replace("`", "").replace("json","")

print(clean_request(sample.content))

req = clean_request(sample.content)

In [ ]:

# Parse the JSON string
steps_data = json.loads(req)

# Create the PDF guide
guide = PDFFile("IKEA Table Assembly Guide")
for step in steps_data:
    guide.add_step(step["step_number"], step["description"], step["image_path"])




NameError: name 'json' is not defined

In [ ]:
# Export the PDF
guide.export_pdf("ikea_table_guide_json.pdf")